# Raptor sibling fine-tune (labels v2)

Fine-tunes the public nartaa Raptor CoAtNet SWA checkpoint (CC0, `nartaa/rsna-knee-publication-swa-weights-20261007`) on the Raptor-96 corpus (built with the inference stage's own preprocessing) and labels v2, with a different seed. Model/windowing from the public Raptor recipe by dreaddevelopment (CC0).

In [ ]:
%%writefile /tmp/train_sib.py
"""Fine-tune a sibling of the public nartaa Raptor CoAtNet (0.949 LB) on labels v2.

Inputs are the Raptor-96 corpus built with the inference stage's own preprocessing (fastread
study_lut, 96 slices, central 320 crop), so the sibling can share that stage's decoded volumes.
Model and windowing follow the public Raptor recipe (dreaddevelopment / nartaa, CC0); changes:
init from the 0.949 SWA checkpoint, fp16 + GradScaler (T4 has no bf16), one seed per GPU, SWA over
the last epochs. Gold-58 stays held out, but the init checkpoint was selected on it, so its
gold AUC is only a sanity check, not a clean estimate.
"""
import os, glob, json, time, random
import psutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
INIT = os.environ.get("INIT", "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt")
START_EPOCH = int(os.environ.get("START_EPOCH", 0))   # >0: INIT is raptor_sib_last.pt of an interrupted run
EPOCHS = int(os.environ.get("EPOCHS", 4))
SWA_LAST = int(os.environ.get("SWA_LAST", 3))
BS_PER_GPU = int(os.environ.get("BS_PER_GPU", 4))
K, K_EVAL = 12, 24
BB_LR, HEAD_LR, WD = float(os.environ.get("BB_LR", 1e-5)), 1e-4, 0.02
SEED = int(os.environ.get("SEED", 7))
LIMIT = int(os.environ.get("LIMIT", 0))            # >0: smoke run on this many studies
HOURS = float(os.environ.get("HOURS", 10.5))       # stop training before the 12 h session limit
OUT = "/kaggle/working"
TAG = os.environ.get("TAG", "")                 # output-name suffix: one process per GPU, one seed each
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True


def find(name):
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d or "abnormality-detection" in d:
            continue
        hits = glob.glob(os.path.join(d, "**", name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)


comp = next(r for r in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",
                        "/kaggle/input/rsna-knee-abnormality-detection") if os.path.exists(r + "/train.csv"))
paths = {}
for d in sorted(glob.glob("/kaggle/input/*/")):
    if "competition" in d or "abnormality-detection" in d:
        continue
    for c in glob.glob(os.path.join(d, "**", "raptor96_c320"), recursive=True):
        for p in glob.glob(c + "/*.npz"):
            paths[os.path.basename(p)[:-4]] = p
print("corpus studies:", len(paths), flush=True)

tr = pd.read_csv(comp + "/train.csv"); tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
gold = tr[tr[LAB].notna().all(axis=1)].set_index("StudyInstanceUID")[LAB].astype(np.float32)
soft = pd.read_parquet(find("labels_llm_soft.parquet")).set_index("StudyInstanceUID")[LAB].astype(np.float32)
gold_ids = [u for u in gold.index if u in paths]
train_ids = [u for u in soft.index if u in paths and u not in gold.index]
if LIMIT:
    train_ids, gold_ids = train_ids[:LIMIT], gold_ids[:max(8, LIMIT // 4)]
labels = {**{u: soft.loc[u].values for u in train_ids}, **{u: gold.loc[u].values for u in gold_ids}}
print(f"train {len(train_ids)} | gold-val {len(gold_ids)}", flush=True)


def centers(mask, k, train):
    D = len(mask)
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = list(range(lo + 1, hi)) or [max(1, min((lo + hi) // 2, D - 2))]
    if train:
        pool = cs * (k // len(cs) + 1); random.shuffle(pool)
        return np.array(pool[:k])
    return np.array([cs[i] for i in np.linspace(0, len(cs) - 1, k).round().astype(int)])


class Bags(Dataset):
    def __init__(self, ids, k, train):
        self.ids, self.k, self.train = ids, k, train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        uid = self.ids[i]
        with np.load(paths[uid]) as z:
            vol, mask = z["vol"], z["mask"]
        cs = np.clip(centers(mask, self.k, self.train), 1, len(mask) - 2)
        x = torch.from_numpy(vol[np.stack([cs - 1, cs, cs + 1], 1)]).float() / 255.0   # (k,3,320,320)
        if self.train:
            x = (x * (1.0 + (random.random() - 0.5) * 0.20)).clamp(0, 1)
        return (x - MEAN) / STD, torch.from_numpy(labels[uid])


class RaptorClassifier(nn.Module):
    """Head of the public Raptor recipe: per-finding attention-MIL over windows."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        self.n = n

    def forward(self, x):
        B, Kw = x.shape[:2]
        h = self.norm(self.backbone(x.flatten(0, 1)).view(B, Kw, -1))
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb


ck = torch.load(find(INIT), map_location="cpu", weights_only=False)
# img_size=320 as in the inference stage (the rmlp position MLP is resolution-free; the 384 checkpoint loads strictly)
bb = timm.create_model(ck.get("arch", ARCH), pretrained=False, num_classes=0, in_chans=3, global_pool="avg", img_size=320)
model = RaptorClassifier(bb, F_dim=bb.num_features)
model.load_state_dict(ck["model"], strict=True)
del ck
bb.set_grad_checkpointing(True)
ngpu = 1   # nn.DataParallel on Kaggle T4 x2 (torch 2.11) leaked ~30 MB host RAM per step and ran 2x slower than one T4
model = model.cuda()
net = model
print(f"init {INIT} | gpus {ngpu} | epochs {EPOCHS} | bs {BS_PER_GPU * max(ngpu, 1)} | bb_lr {BB_LR} | seed {SEED}", flush=True)

workers = int(os.environ.get("WORKERS", 2))
tl = DataLoader(Bags(train_ids, K, True), batch_size=BS_PER_GPU * max(ngpu, 1), shuffle=True,
                num_workers=workers, drop_last=True, pin_memory=True)
vl = DataLoader(Bags(gold_ids, K_EVAL, False), batch_size=2, shuffle=False, num_workers=2)

prev = np.clip(np.stack([labels[u] for u in train_ids]).mean(0), 0.03, 0.7)
pos_weight = torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32).cuda()
lossf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
head = [p for n_, p in model.named_parameters() if not n_.startswith("backbone.")]
opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": BB_LR},
                         {"params": head, "lr": HEAD_LR}], weight_decay=WD)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[BB_LR, HEAD_LR], total_steps=len(tl) * EPOCHS, pct_start=0.05)
scaler = torch.amp.GradScaler("cuda")
for _ in range(len(tl) * START_EPOCH):   # resume the OneCycle schedule where the interrupted run stopped
    sched.step()


@torch.no_grad()
def evaluate(m):
    m.eval(); P, Y = [], []
    for x, y in vl:
        with torch.autocast("cuda", dtype=torch.float16):
            P.append(torch.sigmoid(m(x.cuda(non_blocking=True)).float()).cpu().numpy())
        Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    aucs = {n: roc_auc_score(Y[:, j], P[:, j]) for j, n in enumerate(LAB) if len(set(Y[:, j])) > 1}
    return float(np.mean(list(aucs.values()))), aucs


t0 = time.time()
au0, _ = evaluate(net)
print(f"epoch 0 (init) gold macro AUC {au0:.4f}", flush=True)
snapshots, hist = [], []
for ep in range(START_EPOCH + 1, EPOCHS + 1):
    net.train(); tot, n = 0.0, 0; te = time.time()
    for step, (x, y) in enumerate(tl):
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = net(x)
        loss = lossf(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); n += 1
        if step % 100 == 0:
            el = time.time() - te
            print(f"  ep{ep} step {step}/{len(tl)} loss {tot/n:.4f} | {el/60:.1f} min | "
                  f"ETA epoch {(len(tl)-step-1)*el/max(step+1,1)/60:.0f} min | RSS {psutil.Process().memory_info().rss/2**30:.1f} GB, "
                  f"RAM avail {psutil.virtual_memory().available/2**30:.1f} GB", flush=True)
        if time.time() - t0 > HOURS * 3600:
            break
    au, aucs = evaluate(net)
    hist.append({"ep": ep, "loss": tot / max(n, 1), "gold_auc": au, "minutes": (time.time() - te) / 60})
    print(f"epoch {ep} loss {tot/max(n,1):.4f} gold macro AUC {au:.4f} | {(time.time()-te)/60:.1f} min | "
          + " ".join(f"{k[:6]}={v:.3f}" for k, v in aucs.items()), flush=True)
    snapshots.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    snapshots = snapshots[-SWA_LAST:]
    torch.save({"model": snapshots[-1], "arch": ARCH, "res": 384, "lab": LAB, "epoch": ep}, f"{OUT}/raptor_sib{TAG}_last.pt")
    swa = {k: (sum(s[k].float() for s in snapshots) / len(snapshots)).to(snapshots[-1][k].dtype) for k in snapshots[-1]}
    torch.save({"model": swa, "arch": ARCH, "res": 384, "lab": LAB, "swa_over": [h["ep"] for h in hist][-len(snapshots):]},
               f"{OUT}/raptor_sib{TAG}_swa.pt")
    if time.time() - t0 > HOURS * 3600:
        print("time budget reached", flush=True)
        break

model.load_state_dict(swa)
au_swa, aucs_swa = evaluate(net)
json.dump({"init_gold": au0, "swa_gold": au_swa, "swa_aucs": aucs_swa, "hist": hist, "seed": SEED, "bb_lr": BB_LR,
           "epochs": EPOCHS, "start_epoch": START_EPOCH, "init": INIT, "n_train": len(train_ids)}, open(f"{OUT}/raptor_sib{TAG}.json", "w"), indent=1)
print(f"SWA over {len(snapshots)} epochs: gold macro AUC {au_swa:.4f} (init {au0:.4f}) | total {(time.time()-t0)/60:.0f} min", flush=True)

In [ ]:
import os, subprocess, sys
# Two seeds in parallel, one per T4 (DataParallel leaked host RAM and was slower than one GPU).
base = dict(os.environ, EPOCHS='3', SWA_LAST='2', BS_PER_GPU='4', BB_LR='1e-5', HOURS='10.5', WORKERS='2')
runs = {seed: subprocess.Popen([sys.executable, '-u', '/tmp/train_sib.py'],
                               env={**base, 'CUDA_VISIBLE_DEVICES': gpu, 'SEED': seed, 'TAG': f'_s{seed}'},
                               stdout=open(f'/kaggle/working/sib_s{seed}.log', 'w'), stderr=subprocess.STDOUT)
        for gpu, seed in (('0', '7'), ('1', '11'))}
codes = {seed: p.wait() for seed, p in runs.items()}
for seed in runs:
    print(f"===== seed {seed}: exit {codes[seed]}")
    print("".join(l for l in open(f'/kaggle/working/sib_s{seed}.log') if "step" not in l)[-4000:])
assert all(c == 0 for c in codes.values()), codes